# JOBINSIGHT — AI-POWERED JOB MARKET INTELLIGENCE PLATFORM
## Phase 1, Phase 2 & Phase 3: Preprocessing, EDA, Machine Learning & NLP Pipelines

This notebook represents the complete, reproducible end-to-end pipeline for **JobInsight**:
1. **Data Ingestion & Schema Audit** (LinkedIn Postings Dataset)
2. **Data Cleaning & Missing Value Imputation**
3. **Feature Engineering** (Temporal indices, Role domain classification, Location tiers)
4. **Machine Learning Model Benchmark** (Ridge, HistGradientBoosting, Random Forest, Gradient Boosting)
5. **Evaluation & Metric Selection** (R², MAE, RMSE)
6. **Pipeline Serialization & Export** (Joblib)
7. **Production NLP Entity & Skill Extractor**
8. **FastAPI & Deployment Packaging**

### Section 1: Imports and Environment Configuration

In [ ]:
# Install kagglehub if needed
!pip install -q kagglehub scikit-learn pandas numpy joblib

import os
import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor,
    GradientBoostingRegressor
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')
print('Environment successfully configured!')

### Section 2: Dataset Loading
Loads the LinkedIn Job Postings dataset from Kaggle or Google Drive.

In [ ]:
import kagglehub

# Download official dataset
path = kagglehub.dataset_download('arshkon/linkedin-job-postings')
print('Dataset downloaded to:', path)

csv_file = os.path.join(path, 'postings.csv')
cols = [
    'job_id', 'title', 'company_name', 'normalized_salary', 
    'formatted_work_type', 'formatted_experience_level', 
    'location', 'remote_allowed', 'original_listed_time', 'description'
]
df = pd.read_csv(csv_file, usecols=cols)
print(f'Ingested {len(df):,} job postings with {len(cols)} core features.')

### Section 3: Data Cleaning & Preprocessing

In [ ]:
# 1. Deduplicate by unique primary key
jobs = df.drop_duplicates(subset='job_id').copy()

# 2. Handle missing categorical entries
jobs['company_name'] = jobs['company_name'].fillna('Unknown').str.strip()
jobs['formatted_experience_level'] = jobs['formatted_experience_level'].fillna('Not Specified')
jobs['formatted_work_type'] = jobs['formatted_work_type'].fillna('Full-time')
jobs['remote_allowed'] = jobs['remote_allowed'].fillna(0.0).astype(int).astype(str)
jobs['description'] = jobs['description'].fillna('')

print('Deduplicated Clean Records:', len(jobs))

### Section 4: Feature Engineering
Extracting role categories and geographic cost tiers from unstructured titles and locations.

In [ ]:
def categorize_title(title):
    if not isinstance(title, str):
        return 'General Professional'
    t = title.lower()
    if any(k in t for k in ['software', 'developer', 'engineer', 'frontend', 'backend', 'full stack', 'devops', 'cloud', 'data', 'architect', 'tech', 'machine learning', 'ai']):
        return 'Technology'
    elif any(k in t for k in ['finance', 'accountant', 'accounting', 'controller', 'analyst', 'audit', 'tax', 'banking', 'underwriter']):
        return 'Finance'
    elif any(k in t for k in ['manager', 'director', 'lead', 'product manager', 'project manager', 'scrum', 'program manager', 'vp', 'executive']):
        return 'Management'
    elif any(k in t for k in ['sales', 'account executive', 'business development', 'bdr', 'sdr', 'client', 'representative']):
        return 'Sales'
    elif any(k in t for k in ['nurse', 'therapist', 'medical', 'clinical', 'physician', 'healthcare', 'care', 'dental', 'health']):
        return 'Healthcare'
    elif any(k in t for k in ['operations', 'assistant', 'coordinator', 'specialist', 'administrator', 'receptionist', 'support', 'customer service']):
        return 'Operations'
    elif any(k in t for k in ['retail', 'cashier', 'store', 'associate', 'barista', 'merchandiser', 'restaurant']):
        return 'Retail'
    else:
        return 'General Professional'

def categorize_location(loc):
    if not isinstance(loc, str):
        return 'Tier 3 (Regional / Secondary)'
    l = loc.lower()
    if any(k in l for k in ['ny', 'new york', 'sf', 'san francisco', 'seattle', 'wa', 'boston', 'ma', 'los angeles', 'ca', 'san jose']):
        return 'Tier 1 (NYC, SF, Seattle, Boston)'
    elif any(k in l for k in ['chicago', 'il', 'austin', 'tx', 'dallas', 'houston', 'atlanta', 'ga', 'denver', 'co', 'charlotte', 'nc', 'dc', 'washington']):
        return 'Tier 2 (Austin, Chicago, Denver, Atlanta)'
    else:
        return 'Tier 3 (Regional / Secondary)'

jobs['role_category'] = jobs['title'].apply(categorize_title)
jobs['location_tier'] = jobs['location'].apply(categorize_location)
print('Engineered features: role_category, location_tier.')

### Section 5: Salary Dataset Preparation & Outlier Capping

In [ ]:
# Filter disclosed salaries
salary_df = jobs.dropna(subset=['normalized_salary']).copy()

# Outlier capping between 1st and 99th percentile
p1 = salary_df['normalized_salary'].quantile(0.01)
p99 = salary_df['normalized_salary'].quantile(0.99)
salary_df = salary_df[(salary_df['normalized_salary'] >= max(20000, p1)) & (salary_df['normalized_salary'] <= min(350000, p99))]

feature_cols = ['role_category', 'formatted_experience_level', 'formatted_work_type', 'location_tier', 'remote_allowed']
X = salary_df[feature_cols]
y = salary_df['normalized_salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f'Training on {len(X_train):,} samples | Testing on {len(X_test):,} holdout samples.')

### Section 6 & 7: Model Training, Evaluation & Regression Benchmark

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), feature_cols)
    ]
)

candidate_models = {
    'Ridge Regression': Ridge(alpha=1.0),
    'HistGradientBoosting': HistGradientBoostingRegressor(max_iter=150, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
}

results = {}
pipelines = {}

print('Training Candidate Regression Models...')
for name, model in candidate_models.items():
    pipe = Pipeline([
        ('prep', preprocessor),
        ('reg', model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    pipelines[name] = pipe
    print(f'[{name}] MAE: ${mae:,.0f} | RMSE: ${rmse:,.0f} | R²: {r2:.4f}')

best_name = max(results, key=lambda k: results[k]['R2'])
best_pipe = pipelines[best_name]
print(f'\n>>> Best Performing Algorithm Selected: {best_name}')

### Section 8 & 9: Model Export & Metadata Serialization

In [ ]:
os.makedirs('models', exist_ok=True)
model_path = 'models/salary_model.joblib'
joblib.dump(best_pipe, model_path)

meta_path = 'models/model_metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump({
        'model_name': best_name,
        'metrics': results[best_name],
        'features': feature_cols,
        'training_samples': len(X_train),
        'test_samples': len(X_test)
    }, f, indent=2)

print(f'Pipeline saved to: {model_path} ({os.path.getsize(model_path)/1024:.1f} KB)')
print(f'Metadata saved to: {meta_path}')

### Section 10 & 11: Production NLP Skill Extractor Pipeline Development & Testing

In [ ]:
from backend.app.services.nlp_service import nlp_extractor

sample_desc = '''
We are seeking a Senior Distributed Systems Engineer. 
Must have 5+ years of experience in Python, PyTorch, Go, Docker, and Kubernetes. 
Familiarity with AWS (S3, EC2, Lambda), Terraform, CI/CD pipelines, and Kafka is essential. 
Master's Degree in Computer Science or equivalent. Strong leadership and agile communication skills.
'''

extracted = nlp_extractor.extract(sample_desc)
print('Extracted Domain:', extracted['inferred_category'])
print('Total Entities:', extracted['total_detected'])
print('Technical Skills:', extracted['technical_skills'])
print('Experience:', extracted['experience_required'])
print('Degree Credential:', extracted['degree_required'])

### Section 12: Deployment Preparation
Instructions to start the FastAPI server or deploy the model container to Hugging Face Spaces / Render.

In [ ]:
print('=== JobInsight Machine Learning Pipeline Ready ===')
print('FastAPI Backend Run Command:')
print('cd backend && uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload')
print('Hugging Face Spaces: Deploy backend/ directory with Dockerfile')
print('Production URL: Connected to Vercel via VITE_API_URL')